In [1]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters langchain-google-genai langchain-groq langchain-openrouter langsmith


In [2]:
!pip install yt-dlp faster-whisper chromadb rank_bm25 easyocr

In [3]:
import os

In [4]:
os.getcwd()

'/content'

In [5]:
import yt_dlp
from pydub import AudioSegment
from pathlib import Path


In [6]:
import torch
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [7]:
url='https://www.youtube.com/watch?v=NyOYW07-L5g'

In [8]:
video_dir=os.path.join(os.getcwd(),'videos')
os.makedirs(video_dir, exist_ok=True)

video_name=os.path.join(video_dir, "%(title)s.%(ext)s")

In [9]:
ydl_opts = {
    'format':'best[ext=mp4]',
    'outtmpl':video_name,
    'merge_output_format':'mp4'}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info=ydl.extract_info(url,download=True)
    video_path = ydl.prepare_filename(info)

[youtube] Extracting URL: https://www.youtube.com/watch?v=NyOYW07-L5g
[youtube] NyOYW07-L5g: Downloading webpage


[youtube] NyOYW07-L5g: Downloading android vr player API JSON
[info] NyOYW07-L5g: Downloading 1 format(s): 18
[download] /content/videos/First Law of Thermodynamics, Basic Introduction - Internal Energy, Heat and Work - Chemistry.mp4 has already been downloaded
[download] 100% of   10.31MiB


In [10]:
#Extract audio

In [11]:
wav_path=os.path.join(video_dir,"audio.wav")

In [12]:
video_path

'/content/videos/First Law of Thermodynamics, Basic Introduction - Internal Energy, Heat and Work - Chemistry.mp4'

In [13]:
audio = AudioSegment.from_file(video_path)
audio.export(wav_path,format="wav")

<_io.BufferedRandom name='/content/videos/audio.wav'>

In [14]:
def chunk_audio(wav_path,chunk_min=10):
  audio=AudioSegment.from_wav(wav_path)
  chunk_length=chunk_min*60*1000

  wav_chunk_path=os.path.join(video_dir,"audio_chunks")
  os.makedirs(wav_chunk_path,exist_ok=True)

  chunks=[]
  for i,chunk in enumerate(range(0,len(audio),chunk_length)):
    chunk=audio[chunk:chunk+chunk_length]
    chunk_path=os.path.join(wav_chunk_path,f"{wav_path}_chunk{i}.wav")
    chunk.export(chunk_path,format="wav")
    chunks.append(chunk_path)
  return chunks


In [15]:
audio_chunks=chunk_audio(wav_path,10)

In [16]:
audio_chunks

['/content/videos/audio.wav_chunk0.wav',
 '/content/videos/audio.wav_chunk1.wav']

In [17]:
from faster_whisper import WhisperModel

In [18]:
whisper=WhisperModel("base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [19]:
segments,info=whisper.transcribe(audio_chunks[-1],beam_size=5,vad_filter=True,word_timestamps=True)

In [20]:
segments

<generator object restore_speech_timestamps at 0x7cce2a350280>

In [21]:
info.language

'en'

In [22]:
info.duration

86.173

In [23]:
i=0
for segment in segments:
  i+=1
  print(
      f"[{segment.start:.2f}s -> {segment.end:.2f}s] "
      f"{segment.text}"
      )
  if i==4:
   break

[0.50s -> 9.36s]  heat energy. And so heat energy flows out of the system into the surroundings. And so this is an exothermic process.
[11.74s -> 19.78s]  So anytime heat energy is released, it's exothermic, and when heat energy is absorbed by the system, it's endothermic.
[20.88s -> 26.52s]  So for an exothermic process, Q is negative, and for an endothermic process, Q is positive.
[26.52s -> 39.79s]  And as was mentioned before, W is positive when work is done on the system. This is in chemistry.


In [24]:
def transcribe(chunks):
  transcript_segments=[]
  offset=0
  for chunk in chunks:
    segments,info=whisper.transcribe(chunk,language="en")
    for segment in segments:
      transcript_segments.append({
          "start":segment.start+offset,
          "end":segment.end+offset,
          "text":segment.text
          })
    offset+=info.duration
  return transcript_segments


In [25]:
transcripts=transcribe(audio_chunks)


In [26]:
transcripts[:5]

[{'start': 0.0,
  'end': 11.0,
  'text': " In this video, we're going to talk about the first law of thermodynamics, and how it relates to internal energy, heat, and work."},
 {'start': 11.0,
  'end': 15.0,
  'text': ' So what is the basic idea behind the first law of thermodynamics?'},
 {'start': 15.0,
  'end': 26.0,
  'text': ' The gist of it is this. Energy cannot be created or destroyed. It can simply be transferred from one place to another.'},
 {'start': 27.0,
  'end': 36.0,
  'text': " So let's say if we have a system, and everything outside of it is the surroundings."},
 {'start': 42.0,
  'end': 51.0,
  'text': " Energy can flow into or out of the system, and there's two ways in which energy can do so, and that is through heat and work."}]

In [27]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [28]:
def format_time(seconds):
  hours=int(seconds//(60*60))
  minutes=int((seconds%(60*60))//60)
  seconds=int(seconds%60)
  return f"{hours:02d}:{minutes:02d}:{seconds:02d}" if hours>0 else f"{minutes:02d}:{seconds:02d}"


In [29]:
text_docs=[]

curr_text=""
start_time=None
end_time=None

chunk_thershold=1000
chunk_id=0;

for segment in transcripts:
  if start_time is None:
    start_time=segment['start']
  curr_text+=segment['text']

  if len(curr_text)>=chunk_thershold:
    end_time=segment['end']
    doc=Document(page_content=curr_text,
               metadata={
                   'chunk_id':chunk_id,
                   'start':format_time(start_time), 'end':format_time(end_time),
                   'start_sec': start_time, 'end_sec':end_time
               })
    text_docs.append(doc)
    curr_text=""
    start_time=None
    chunk_id+=1;

if curr_text.strip():
  end_time=transcripts[-1]['end']
  doc=Document(page_content=curr_text,
               metadata={
                   'chunk_id':chunk_id,
                   'start':format_time(start_time), 'end':format_time(end_time),
                   'start_sec': start_time, 'end_sec':end_time
               })
  text_docs.append(doc)

In [30]:
text_docs[0]

Document(metadata={'chunk_id': 0, 'start': '00:00', 'end': '01:44', 'start_sec': 0.0, 'end_sec': 104.0}, page_content=" In this video, we're going to talk about the first law of thermodynamics, and how it relates to internal energy, heat, and work. So what is the basic idea behind the first law of thermodynamics? The gist of it is this. Energy cannot be created or destroyed. It can simply be transferred from one place to another. So let's say if we have a system, and everything outside of it is the surroundings. Energy can flow into or out of the system, and there's two ways in which energy can do so, and that is through heat and work. So if heat flows into the system, the system gains energy, and that energy is known as the internal energy of the system, represented by the symbol capital U. Now the surroundings can do work on a system, so those are the two ways in which a system can increase its internal energy, is by the transfer of heat energy into the system, or if the surroundings

In [31]:
# Skipping this  bec no point to chunk once again bec alredy did
# splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
# split_docs = splitter.split_documents(docs)

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


/tmp/ipykernel_6005/2817082483.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [33]:
text_embedding_model=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [34]:
text_db_path=os.path.join(os.getcwd(),'db')

text_db = Chroma.from_documents(
    documents=text_docs,
    embedding=text_embedding_model,
    persist_directory=text_db_path
)

In [35]:

# text_db = Chroma(
#     persist_directory="./chroma_db",
#     embedding_function=embeddings
# )

In [36]:
#Extract frames

In [37]:
import cv2
import numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from collections import deque


In [38]:
from sentence_transformers import SentenceTransformer

In [39]:
clip_model=SentenceTransformer('clip-ViT-B-32')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [40]:
def extract_changed_frames(video_path, threshold=15.0, min_gap_sec=1.0,
                           clip_similarity_threshold=0.95, history_size=20):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    min_gap_frames = int(fps * min_gap_sec)

    scores = []
    frames = []
    timestamps = []

    frame_idx = 0
    anchor_gray = None          # last KEPT frame, not last seen frame
    last_kept_idx = -min_gap_frames
    recent_embeddings = deque(maxlen=history_size)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if anchor_gray is None:
            anchor_gray = gray
            frame_idx += 1
            continue

        score = np.mean(cv2.absdiff(anchor_gray, gray))
        scores.append(score)

        if score > threshold and (frame_idx - last_kept_idx) >= min_gap_frames:
            timestamp = frame_idx / fps
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)

            embedding = clip_model.encode(pil_image, convert_to_numpy=True, normalize_embeddings=True)

            keep = True
            for prev_embedding in recent_embeddings:
                sim = cosine_similarity(embedding.reshape(1, -1), prev_embedding.reshape(1, -1))[0][0]
                if sim > clip_similarity_threshold:
                    keep = False
                    break

            if keep:
                frames.append(pil_image)
                timestamps.append(timestamp)
                recent_embeddings.append(embedding)
                last_kept_idx = frame_idx

            anchor_gray = gray   # reset anchor either way — accumulated drift has been consumed

        frame_idx += 1

    cap.release()
    print(f"Min:{min(scores):.3f} Max:{max(scores):.3f} Mean:{np.mean(scores):.3f} "
          f"P90:{np.percentile(scores,90):.3f} P99:{np.percentile(scores,99):.3f}")
    return frames, timestamps

In [41]:
def calibrate_threshold(video_path, sample_every=5, percentile=92):
    cap = cv2.VideoCapture(video_path)
    prev_gray = None
    diffs = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_every == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if prev_gray is not None:
                diffs.append(np.mean(cv2.absdiff(prev_gray, gray)))
            prev_gray = gray
        idx += 1
    cap.release()
    diffs = np.array(diffs)
    print(f"frame-to-frame noise floor — median:{np.median(diffs):.3f} p95:{np.percentile(diffs,95):.3f}")
    return max(2.0, np.percentile(diffs, percentile) * 4)   # scale factor is the only thing you tune

In [42]:
threshold = calibrate_threshold(video_path)
frames, frame_timestamps = extract_changed_frames(video_path, threshold=threshold, min_gap_sec=1.0,
                                                   clip_similarity_threshold=0.95)

frame-to-frame noise floor — median:0.001 p95:0.071
Min:0.000 Max:11.798 Mean:0.793 P90:1.761 P99:1.922


In [43]:
len(frames)

12

In [44]:
frames[:2]

[<PIL.Image.Image image mode=RGB size=640x360>,
 <PIL.Image.Image image mode=RGB size=640x360>]

In [45]:
#normaizing bec ->Cosine similarity doesnt effect on normalization(i.e norm) but still do bec it make computation fast

clip_embedding=clip_model.encode(frames,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [46]:
#Is it neccesary after already doing-> compare last n embedding?(Check)

#  dedup: throw away near-identical consecutive frames
# input: embeddings | output: a smaller list of frames that survive
# kept_frames, kept_embeddings, kept_timestamps = [], [], []
# for i, emb in enumerate(clip_embedding):
#     if not kept_embeddings or cosine_sim(emb, kept_embeddings[-1]) < 0.93:
#         kept_frames.append(frames[i])
#         kept_embeddings.append(emb)
#         kept_timestamps.append(frame_timestamps[i])

In [47]:
import easyocr

reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

def ocr_frame(frame_pil):
    # Convert BGR to RGB for EasyOCR
    frame_rgb=cv2.cvtColor(np.array(frame_pil),cv2.COLOR_BGR2RGB)
    # Perform OCR
    result=reader.readtext(frame_rgb,detail=0)  # detail=0 returns only text
    return ' '.join(result)


In [48]:
frames_dir = os.path.join(os.getcwd(), 'frame_images')
os.makedirs(frames_dir, exist_ok=True)

In [49]:
frame_docs=[]
chunk_idx=0

for i ,(frame,ts) in enumerate(zip(frames,frame_timestamps)):
  while chunk_idx < len(text_docs)-1 and ts >=text_docs[chunk_idx].metadata['end_sec']:
    chunk_idx+=1

  frame_path=os.path.join(frames_dir,f"frame_{i}.jpg")
  frame.save(frame_path)

  ocr_text=ocr_frame(frame)

  frame_docs.append(Document(
      page_content=f"Frame {i} at {ts:.2f} seconds",
      metadata={
          'frame_path':frame_path,
          'ocr_text':ocr_text,
          'chunk_id':text_docs[chunk_idx].metadata['chunk_id'],
          'frame_idx':i,
          'frame_ts':ts
      }
    ))


In [50]:
import chromadb

In [51]:
frame_db_path=os.path.join(os.getcwd(),'frame_db')

client=chromadb.PersistentClient(path=frame_db_path)

#By default chromadb use l2 similarity
frame_db=client.get_or_create_collection("frames",metadata={"hnsw:space":"cosine"})

In [52]:
frame_db.add(
    documents=[doc.page_content for doc in frame_docs],
    metadatas=[doc.metadata for doc in frame_docs],
    ids=[f"frame_{i}" for i in range(len(frame_docs))],
    embeddings=clip_embedding.tolist()
)

In [53]:
import langsmith
print(langsmith.__version__)

0.8.15


In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env")

In [56]:
from pydantic import BaseModel, Field
from typing import Literal

In [57]:
class Answer(BaseModel):
    response: str=Field(
        """
        A clear educational explanation written in the model's own words.
        Do not quote transcript verbatim unless necessary"""
        )

    timestamps: list[str]=Field(
        default_factory=list,
        description="Relevant timestamp ranges from the video. Empty list if answered from general knowledge only."
    )

    source: Literal["video","general_knowledge","hybrid"]=Field(
        description=(
            "'video'=answered from transcript context. "
            "'general_knowledge'=video didn't cover it, used prior knowledge. "
            "'hybrid'=used both."
        )
    )

    key_takeway: list[str]=Field(
        description="Point out the important point only"
    )


In [58]:
text_retriever=text_db.as_retriever(search_type='similarity',search_kwargs={'k':20}) #other option mmr try later
#

In [59]:
query="Summarize the video and give important points"

In [60]:
text_retriever.invoke(query)[0]

Document(metadata={'end': '03:06', 'end_sec': 186.0, 'start_sec': 105.0, 'start': '01:45', 'chunk_id': 1}, page_content=" The energy of the surroundings, however, has a decrease by 300, and so energy is not created or destroyed, is simply transferred from one place to another. The system, didn't just get the 300 joules from nowhere, that 300 joules of energy came from somewhere, it came from the surroundings. A good way to illustrate this is to use money. So let's say if you decide to sell a laptop for $500, someone will buy the laptop for $500, let's say if someone does buy it. Your bank account will increase by $500, but that person's bank account will decrease by $500, and so the money just doesn't come from nowhere, just magically appears. It comes from somewhere, so in order for you to gain $500, someone else has to lose $500. Now granted, the government could print more money if they wanted to, but in a practical sense, in everyday life, when someone gains money, someone has to l

In [61]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(text_docs)

bm25_retriever.k = 20

In [62]:
bm25_retriever.invoke(query)[0]

Document(metadata={'chunk_id': 7, 'start': '10:31', 'end': '11:04', 'start_sec': 631.92, 'end_sec': 664.48}, page_content=" W is positive when work is done on the system. This is in chemistry. And W is negative when work is done by the system. So, you need to be familiar with the sine conventions if you're going to use this equation. So, in another video, I'm going to give you some practice problems on calculating the change in internal energy using heat and work.")

In [63]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever=EnsembleRetriever(
    retrievers=[bm25_retriever,text_retriever],
    weights=[0.3, 0.7])

In [64]:
from sentence_transformers import CrossEncoder

reranker=CrossEncoder("BAAI/bge-reranker-base")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [65]:
def rerank(query,docs,k=5):

  pairs=[(query,doc.page_content) for doc in docs]

  scores=reranker.predict(pairs)
  reranked=sorted(zip(docs,scores),key=lambda x:x[1],reverse=True)

  return [doc for doc,score in reranked[:k]]

In [66]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage



In [67]:
#Ignore It->
# You are an expert educational tutor helping a student understand video lecture content.

# ## Your Knowledge Sources (in priority order)
# 1. The video transcript provided below
# 2. OCR text extracted from relevant video frames (if available)
# 3. Retrieved video frames/images (if attached)
# 4. Your general educational knowledge

# ## How to Use Each Knowledge Source
# - Use the transcript to understand what the instructor explained.
# - Use OCR text to read slide text, equations, formulas, code snippets, labels, tables, or captions.
# - Use the retrieved video frames to understand visual information if OCR alone cannot capture.
# - Combine all available sources whenever they complement each other.

# ## Critical Rules
# - If the video fully covers the topic:
#   - Explain it in your own words.
#   - Reference the lecture naturally (e.g. "The lecture explains...", "Around timestamp X...").

# - If the video partially covers the topic:
#   - Clearly distinguish what comes from the lecture and what comes from your general knowledge.

# - If the video does not cover the topic:
#   - Say:
#     "The video doesn't address this directly, but..."
#   - Then provide a clear explanation using your general knowledge.

# - If there is any contradiction between the information from video and from your knowledge specify it.
# - If transcript and visual information complement each other, use both.
# - If visual evidence clarifies something that is difficult to understand from the transcript alone, use the visual evidence to improve the explanation.

# ## How to Answer Based on Question Type

# - Summary request:
#   Extract the key ideas and organize them into a coherent explanation instead of copying transcript sentences.

# - Concept explanation:
#   Define the concept clearly, explain it step by step, and include examples or analogies whenever helpful.

# - Factual question:
#   Answer directly and reference the relevant timestamp whenever possible.

# - Questions about slides, diagrams, graphs, equations, code, demonstrations, or anything shown visually:
#   Use the retrieved frames (and OCR if available) together with the transcript.

# - Follow-up questions:
#   Continue naturally from the previous conversation and provide a deeper explanation when appropriate.

# ## Educational Style
# - Explain complex topics step by step.
# - Prefer clarity over technical jargon.
# - Use examples and analogies whenever they improve understanding.
# - For technical definitions, be precise even if the lecture was informal.
# - Always prioritize helping the student understand the material.

# ## Available Context

# ### Video Transcript
# {context}

# ### OCR Text from Retrieved Frames
# {ocr_context}

# ### Retrieved Video Frames
# If images are attached with this request, they are frames extracted from the lecture and are relevant to the user's question. Use them only as supporting visual evidence.


In [68]:
from langchain_google_genai import ChatGoogleGenerativeAI

# llm=ChatOpenRouter(model="nvidia/nemotron-nano-12b-v2-vl:free")
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

strutured_llm=llm.with_structured_output(Answer)

In [69]:
qa_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an expert educational tutor helping a student understand video lecture content.

## Your Knowledge Sources (in priority order)
1. The video transcript provided below
2. OCR text extracted from relevant video frames (if available)
3. Retrieved video frames/images (if attached)
4. Your general educational knowledge

## How to Use Each Knowledge Source
- Use the transcript to understand what the instructor explained.
- Use OCR text to read slide text,etc.
- Use the retrieved video frames to understand visual information if OCR alone cannot capture.
- Combine all available sources whenever they complement each other.

## Critical Rules
- If the video fully covers the topic:
  - Explain it in your own words unless specify.
  - Reference the lecture naturally (e.g. "The lecture explains...").

- If the video partially covers the topic:
  - Clearly distinguish what comes from the lecture and what comes from your general knowledge.

- If the video does not cover the topic:
  - Say:
    "The video doesn't address this directly, but..."
  - Then provide a clear explanation using your general knowledge.

- If there is any contradiction between the information from video and from your knowledge specify it.

## How to Answer Based on Question Type

## Educational Style
- Explain complex topics step by step.
- Use examples and analogies whenever they improve understanding.
- For technical definitions, be precise even if the lecture was informal.
- Always prioritize helping the student understand the material.

## Available Context

### Video Transcript
{context}

### OCR Text from Retrieved Frames
{ocr_context}

### Retrieved Video Frames
If images are attached with this request, they are frames extracted from the lecture and are relevant to the user's question.
"""
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{query}")
])

In [70]:
contextualize_qa_prompt=ChatPromptTemplate.from_messages([
    ("system","""
      You are ONLY a query rewriter.
      Your task is to rewrite the user's last question into a standalone question.
      Rules:
      - NEVER answer the question.
      - NEVER explain.
      - Output ONLY the rewritten question.
      - If the question is already standalone, return it unchanged.

      Examples

      History:
      User: Explain thermodynamics.

      Question:
      What is system?

      Output:
      What is the meaning of "system" in thermodynamics?

    """),
    MessagesPlaceholder("chat_history"),
    ("human","{query}")
])

In [92]:
contextualize_chain=contextualize_qa_prompt|llm|StrOutputParser()

In [71]:
chat_history=InMemoryChatMessageHistory()

In [72]:
def build_text_context(docs):
  formatted=[] #using list bec string is immutable so every time it create new string if we just add

  for doc in docs:
      context= f"""
      Timestamp: {doc.metadata['start']} - {doc.metadata['end']}
      Content: {doc.page_content}
      """
      formatted.append(context)

  return "\n\n".join(formatted)

In [73]:
import tiktoken
from difflib import SequenceMatcher

enc = tiktoken.get_encoding("cl100k_base")

In [74]:
def is_similar(text,seen_texts,threshold=0.85):

  for seen_text in seen_texts:
    ratio=SequenceMatcher(None,text,seen_text).ratio()
    if ratio>threshold:
      return True
  return False

In [75]:
def build_ocr_context(frames_metadata,max_token=300):

  if not frames_metadata:
    return "No relevant on-screen text found for this query"

  ocr,seen=[],[]
  total_token=0

  for frame in frames_metadata:
    ocr_text=frame['ocr_text']

    if not ocr_text or is_similar(ocr_text,seen):
      continue

    block=f"Frame: {frame['frame_ts']}\nOCR Text: {ocr_text}"
    block_token=len(enc.encode(block))

    if total_token+block_token>max_token:
      break

    ocr.append(block)
    seen.append(ocr_text)
    total_token+=block_token

  return "\n\n".join(ocr)




In [76]:
import base64

def load_images(frames_metadata):
  images=[]
  for frame in frames_metadata:
    path=frame['frame_path']
    with open(path,"rb") as f:
            images.append(base64.b64encode(f.read()).decode())
  return images

In [77]:
class Summary(BaseModel):
  question: str = Field(description="The reformulated standalone question, NOT an answer")

In [78]:
from langchain_openrouter import ChatOpenRouter

summarize_llm=ChatOpenRouter(model="meta-llama/llama-3.3-70b-instruct:free")

In [79]:
summarize_prompt=ChatPromptTemplate.from_template(
    """
    Previous summary (may be empty):
  {summary}

  Messages to fold into the summary:
  {new_messages}

  Produce one updated summary capturing all important context an LLM
  would need to continue this conversation. Keep it concise
    """
)

In [80]:
chat_summary=""

In [81]:
def update_summary(messages):
  global chat_summary

  chain=summarize_prompt|llm|StrOutputParser()

  formatted = "\n".join( f"{msg.type}: {msg.content}" for msg in messages)

  chat_summary=chain.invoke({
      "summary":chat_summary,
      "new_messages":formatted
  })
  return chat_summary


In [82]:
def build_multimodal_message(text_context,ocr_context,standalone_ques,images,provider="gemini"):
  prompt_value=qa_prompt.invoke({
      "context":text_context,
      "ocr_context":ocr_context,
      "query":standalone_ques,
      "chat_history":chat_history.messages
  })

  messages=prompt_value.messages

  if len(images)>0:
        content = [{
                "type": "text",
                "text": messages[-1].content,
            }]

        for img in images:
          if provider=="openrouter":
            content.append({
                "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img}"
                    }
            })

          elif provider=="gemini":
            content.append({
                "type":"image",
                "source_type":"base64",
                "mime_type":"image/jpeg",
                "data":img
            })

        messages[-1]=HumanMessage(content=content)

  return messages

In [83]:
class Decision(BaseModel):
  answer: bool

In [84]:
from langchain_groq import ChatGroq

decision_llm=ChatGroq(model="llama-3.1-8b-instant",temperature=0)
decision_strutured_llm=decision_llm.with_structured_output(Decision)

In [85]:
def query_needs_images(query):
  decision_prompt = ChatPromptTemplate.from_template("""
  Determine whether the following question strictly requires video frames.
  Question:
  {query}
  """)
  decision_chain=decision_prompt|decision_strutured_llm
  decision=decision_chain.invoke({"query":query})
  return decision.answer


In [86]:
import time

In [93]:
def chat(query):

  if chat_history.messages:
    standalone_ques=contextualize_chain.invoke({
        "query":query,
        "chat_history":chat_history.messages
    })
  else:
    standalone_ques=query
  print("StandAlone Ques:\n",standalone_ques)

  docs=ensemble_retriever.invoke(standalone_ques)

  docs=rerank(standalone_ques,docs,k=3)

  text_context=build_text_context(docs)

  chunk_ids=[doc.metadata['chunk_id'] for doc in docs]

  query_embedding=clip_model.encode(standalone_ques,convert_to_numpy=True,normalize_embeddings=True)

  frames=frame_db.query(
      query_embeddings=[query_embedding.tolist()],
      where={"chunk_id": {"$in": chunk_ids}},
      n_results=3,
      include=["metadatas","distances"]
  )
  print("No of Frames:",len(frames))

  frames_metadata=frames['metadatas'][0]

  ocr_context=build_ocr_context(frames_metadata)
  print("Length of OCR Contex:",len(ocr_context))


  needs_vision = query_needs_images(standalone_ques)
  print("Needs Vision?:",needs_vision)

  images=[]
  if needs_vision:
    images=load_images(frames_metadata)
    print("Length of load images",len(images))


  # qa_chain=qa_prompt|strutured_llm
  # result=qa_chain.invoke({
  #     "context":context,
  #     "query":query,
  #     "chat_history":chat_history.messages
  # })

  messages=build_multimodal_message(text_context,ocr_context,standalone_ques,images)
  print("Length of messages:",len(messages))

  try:
    print("Calling LLM")

    start = time.time()
    result=strutured_llm.invoke(messages)
    end = time.time()

    print("Time taken:",end-start)
  except Exception as e:
    print(type(e))
    print(e)
    raise

  MAX_TURNS = 3

  if len(chat_history.messages)>MAX_TURNS*2:
    old_messages=chat_history.messages[:-(MAX_TURNS*2)]
    update_summary(old_messages)
    del chat_history.messages[:-(MAX_TURNS*2)]

  chat_history.add_user_message(query)
  chat_history.add_ai_message(result.response)

  return result



In [88]:
# chat_history.clear()

In [89]:
r1=chat("Explain the thermodynamics in short")
print(r1)

StandAlone Ques:
 Explain the thermodynamics in short
No of Frames: 8
Length of OCR Contex: 173
Needs Vision?: False
Length of messages: 2
Calling LLM
Time taken: 5.098278284072876
response="Thermodynamics, as explained in the video, primarily focuses on the First Law of Thermodynamics, which states that energy cannot be created or destroyed, only transferred. This transfer occurs between a 'system' (e.g., reactants and products) and its 'surroundings'. The total energy within a system is called its internal energy, represented by 'U'.\n\nEnergy can be transferred in two main ways: as heat (Q) or as work (W). The change in internal energy (ΔU) of a system is given by the equation: ΔU = Q + W.\n\nHere's how the signs work from the system's perspective:\n*   **Heat (Q)**: Q is positive when the system absorbs heat (an endothermic process), meaning heat flows into the system. Q is negative when the system releases heat (an exothermic process), meaning heat flows out of the system.\n*   **

In [94]:
print(len(r1.response))

1017


In [95]:
r2=chat("what do you mean by system in above answer")
print(r2)

StandAlone Ques:
 What is the meaning of "system" in the context of the thermodynamics explanation?
No of Frames: 8
Length of OCR Contex: 242
Needs Vision?: False
Length of messages: 4
Calling LLM
Time taken: 1.7640788555145264
response='In the context of thermodynamics, as explained in the video, a "system" refers to the specific part of the universe that you are studying or interested in. Everything outside of this defined system is considered the "surroundings." Energy, in the form of heat or work, can be transferred into or out of this system.' timestamps=['00:00 - 01:44'] source='video' key_takeway=["A 'system' is the specific part of the universe under study.", "Everything outside the system is the 'surroundings'.", 'Energy (heat and work) can be transferred between the system and surroundings.']


In [96]:
r3=chat("Can you summarize the whole video in points")
print(r3)

StandAlone Ques:
 Can you summarize the whole video in points?
No of Frames: 8
Length of OCR Contex: 212
Needs Vision?: True
Length of load images 3
Length of messages: 6
Calling LLM
Time taken: 3.4667129516601562
response="Here's a summary of the video's key points:\n\n*   **First Law of Thermodynamics**: Energy cannot be created or destroyed; it can only be transferred from one place to another.\n*   **System and Surroundings**: A 'system' is the part being studied, and everything else is the 'surroundings'. Energy transfer occurs between them.\n*   **Internal Energy (U)**: This is the total energy within a system. A change in internal energy (ΔU) indicates an energy transfer.\n*   **Ways to Transfer Energy**: Energy can flow into or out of a system in two forms:\n    *   **Heat (Q)**\n    *   **Work (W)**\n*   **First Law Equation**: The change in internal energy (ΔU) is the sum of heat (Q) and work (W): ΔU = Q + W.\n*   **Sign Conventions (from the system's perspective)**:\n    *  

In [97]:
import json

# Evalution

In [98]:
#Evaluating retrievral

In [104]:
from langchain_openrouter import ChatOpenRouter

eval_llm=ChatOpenRouter(model="openai/gpt-oss-120b:free")
# eval_llm=ChatOpenRouter(model="openai/gpt-oss-20b:free")

In [100]:
class QAPairs(BaseModel):
  question: str
  answer: str

class QAPairsList(BaseModel):
  pairs: list[QAPairs]

In [105]:
def generate_qa_pairs(docs,n=2):
  qa_pairs=[]
  generation_prompt = PromptTemplate.from_template(
      """You are creating an evaluation dataset for an educational video assistant.

  Given this transcript segment from a lecture video:
  {content}

  Generate {n} question-answer pairs where:
  - The question is something a student would genuinely ask
  - The answer must be completely answerable using only the information contained in this segment.
  - Do not generate questions that require information from previous or subsequent transcript segments.
  - Questions should test understanding, not verbatim recall
  - Mix types: conceptual ("what is X"), factual ("what did the speaker say about X"), and explanatory ("how does X work")

  Return ONLY a JSON array, no other text:
  [{{"question": "...", "answer": "..."}}]"""
  )

  structured_eval_llm=eval_llm.with_structured_output(QAPairsList)
  for doc in docs:
    chain=generation_prompt|structured_eval_llm

    response=chain.invoke({"content":doc.page_content,"n":n})

    #Assuming The chunk that generated the question is the ground-truth chunk.(Not perfect but simplify)

    for pair in response.pairs:
      qa_pairs.append({
      # "source_id":doc.metadata["source"], for multivideo ai assistant
      "question": pair.question,
      "answer": pair.answer,
      "start": doc.metadata["start"],
      "end": doc.metadata["end"],
      "content": doc.page_content
      })
  return qa_pairs

In [108]:
import random

eval_docs=random.sample(text_docs, min(50,len(text_docs)//2))
try:
    qa_pairs = generate_qa_pairs(eval_docs, n=2)
except Exception as e:
    print(type(e))
    print(e)
    raise


<class 'openrouter.errors.toomanyrequestsresponse_error.TooManyRequestsResponseError'>
Provider returned error


TooManyRequestsResponseError: Provider returned error

In [ ]:
print(f"Length:{len(qa_pairs)} from docs of length {len(eval_docs)}")

In [ ]:
#Check multiquery Retrieval

In [ ]:
def multiquery_retrieve(question,retriever):
  query_prompt=PromptTemplate(
    template="""
    You are helping retrieve video transcript chunks.

    Generate 2 alternative queries that may use
    different terminology but ask the same thing.

    Question:
    {question}

    Return one query per line.
    """,
        input_variables=["question"]
  )

  query_chain=query_prompt|eval_llm|StrOutputParser()

  response=query_chain.invoke({"question": question})

  queries=[question] + response.split("\n")

  all_docs=[]

  for q in queries:
      docs=retriever.invoke(q)
      all_docs.extend(docs)

  return all_docs


def remove_duplicates(docs):

    seen = set()
    unique_docs = []

    for doc in docs:
        key = doc.page_content

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs

In [ ]:
def evaluate_retrieval(qa_pairs, retrieval_fn, k_values=[1, 3, 5]):

    hit_counts = {k: 0 for k in k_values}
    reciprocal_ranks = []

    for qa in qa_pairs:
        query = qa["question"]
        source_start = qa["start"]

        retrieved = retrieval_fn(query)
        retrieved_starts = [doc.metadata["start"] for doc in retrieved]

        #With one groutnd truth cunk recall@k=hit@k

        # Hit Rate@k
        for k in k_values:
            if source_start in retrieved_starts[:k]:
                hit_counts[k] += 1

        # MRR
        if source_start in retrieved_starts:
            rank = retrieved_starts.index(source_start) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    n = len(qa_pairs)
    hit_rates = {k: hit_counts[k] / n for k in k_values}
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return {"hit_rate": hit_rates, "mrr": mrr}

In [ ]:
def bm25_only(query):
    return bm25_retriever.invoke(query)

def vector_only(query):
    return text_retriever.invoke(query)

def ensemble_only(query):
    return ensemble_retriever.invoke(query)


# Configuration 4: Full pipeline (ensemble + reranker) — current system
def full_pipeline(query):
    docs = ensemble_retriever.invoke(query)
    return rerank(query,docs,k=5)  # returns top 5

def multiquery_full_pipeline(query):
    docs=multiquery_retrieve(query,ensemble_retriever)
    docs=remove_duplicates(docs)
    docs=rerank(query,docs,k=5)
    return docs  # returns top 5


In [ ]:
configs = {
    "bm25_only":bm25_only,
    "vector_only":vector_only,
    "ensemble_only":ensemble_only,
    "ensemble_rerank":full_pipeline,
    "ensemble_multiquery_rerank":multiquery_full_pipeline
    # Add more configurations as needed
}

retrieval_results = {}
for name, fn in configs.items():
    result = evaluate_retrieval(qa_pairs, fn, k_values=[1, 3, 5])
    retrieval_results[name] = result
    print(f"\n{name}")
    print(f"  MRR:          {result['mrr']:.3f}")
    print(f"  Hit Rate@1:   {result['hit_rate'][1]:.3f}")
    print(f"  Hit Rate@3:   {result['hit_rate'][3]:.3f}")
    print(f"  Hit Rate@5:   {result['hit_rate'][5]:.3f}")

In [ ]:
class JudgeScore(BaseModel):
    correctness: int
    completeness: int
    faithfulness: int
    clarity: int
    reasoning: str

In [ ]:
def judge_answer(question,reference_answer,generated_answer,content,llm):

    judge_prompt = PromptTemplate(
    template="""You are evaluating an AI tutor's answer for an educational video assistant.

    Question: {question}

    Reference Answer (from transcript): {reference_answer}

    Source Transcript (what was retrieved): {content}

    Generated Answer: {generated_answer}

    Rate the Generated Answer on each dimension from 1 to 5:
    - correctness: Is the information factually accurate compared to the reference?
    - completeness: Does it cover the key points from the reference?
    - faithfulness: Does it avoid adding false information not in the source?
    - clarity: Is it clearly explained for a student?
    """,
      input_variables=[
        "question",
        "reference_answer",
        "generated_answer",
        "content"]
    )

    judge_llm=llm.with_structured_output(JudgeScore)
    chain=judge_prompt|judge_llm
    response=chain.invoke({
       "question": question,
       "reference_answer": reference_answer,
       "generated_answer": generated_answer,
       "content": content
    })

    return response.model_dump()


def evaluate_answers(qa_pairs, chat_fn, llm):

    all_scores = []
    failures = []

    for qa in qa_pairs:
        try:
            result = chat_fn(qa["question"])
            generated = result.response

            scores = judge_answer(
                question=qa["question"],
                reference_answer=qa["reference_answer"],
                generated_answer=generated,
                content=qa["content"],
                llm=llm
            )


            scores["question"] = qa["question"]
            scores["source"] = result.source
            all_scores.append(scores)

        except Exception as e:
            failures.append({"question": qa["question"], "error": str(e)})
            print(f"Error evaluating question: {qa['question']}")
            print(e)

    # Aggregate
    dims = ["correctness", "completeness", "faithfulness", "clarity"]
    avg_scores = {d: sum(s[d] for s in all_scores) / len(all_scores) for d in dims}
    avg_scores["overall"] = sum(avg_scores.values()) / len(dims)

    print(f"\nAnswer Quality Results ({len(all_scores)} evaluated, {len(failures)} failed)")
    for dim, score in avg_scores.items():
        print(f"  {dim}: {score:.2f} / 5.0")

    return all_scores, avg_scores, failures

In [ ]:
chat_history.clear()

answer_scores,avg_scores,failures=evaluate_answers(qa_pairs,chat,eval_llm)

chat_history.clear()

In [ ]:
#Evaluating the reranker

In [ ]:
def get_rank(target_start,docs):
  start=[doc.metadata['start'] for doc in docs]
  return start.index(target_start)+1 if target_start in start else None

In [ ]:
def compute_mrr(ranks):
    scores = []

    for rank in ranks:
        if rank is None:
            scores.append(0.0)
        else:
            scores.append(1.0 / rank)

    return sum(scores) / len(scores)


def hit_rate_at_k(ranks, k):
    hits = sum(
        1 for rank in ranks
        if rank is not None and rank <= k
    )

    return hits / len(ranks)


In [ ]:
def evaluate_reranker(qa_pairs,retriever,reranker):
  results=[]

  for qa in qa_pairs:

      query=qa["question"]
      target_start=qa["start"]

      # Before reranking
      pre_docs=retriever.invoke(query)

      pre_rank=get_rank(target_start,pre_docs)

      # After reranking
      pairs=[(query, doc.page_content) for doc in pre_docs]

      scores=reranker.predict(pairs)

      reranked=sorted(zip(pre_docs,scores),key=lambda x:x[1],reverse=True)

      post_docs=[doc for doc, score in reranked]

      post_rank=get_rank(target_start,post_docs)

      movement=None

      if pre_rank is not None and post_rank is not None:
          movement=pre_rank-post_rank

      results.append({
          "question": query,
          "pre_rank": pre_rank,
          "post_rank": post_rank,
          "movement": movement
      })


  pre_ranks=[r["pre_rank"] for r in results]
  post_ranks=[r["post_rank"] for r in results]

  print("\n===== Retrieval vs Reranker =====")

  print(f"MRR Before: {compute_mrr(pre_ranks):.3f}")
  print(f"MRR After : {compute_mrr(post_ranks):.3f}")

  for k in [1, 3, 5]:
      print(f"Hit@{k} Before: "f"{hit_rate_at_k(pre_ranks,k):.3f}")

      print(f"Hit@{k} After : "f"{hit_rate_at_k(post_ranks,k):.3f}")

  valid_moves=[ r["movement"] for r in results if r["movement"] is not None]

  improved=sum( 1 for m in valid_moves if m > 0)

  worsened=sum( 1 for m in valid_moves if m < 0)

  unchanged=sum( 1 for m in valid_moves if m == 0)

  print("\nRank Movement")

  print(f"Improved : {improved}")
  print(f"Worsened : {worsened}")
  print(f"Unchanged: {unchanged}")

  if valid_moves:
      avg_move=sum(valid_moves) / len(valid_moves)

      print(f"Average movement: "f"{avg_move:+.2f}")

  return results


In [ ]:
rank_changes = evaluate_reranker(qa_pairs,ensemble_retriever,reranker)

In [ ]:
worsened_cases = [
    r for r in rank_changes
    if r["movement"] is not None
    and r["movement"] < 0
]
print(len(worsened_cases))

for case in worsened_cases:
    print(case)